In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_ROOT = Path("../../data")
USER_FILE = DATA_ROOT / "raw" / "anime_user_filtered" / "user-filtered.csv"
ANIME_FILE = DATA_ROOT / "processed" / "anime.parquet"
OUTPUT_FILE = DATA_ROOT / "processed" / "interactions_clean.parquet"

CHUNK_SIZE = 5_000_000
MIN_USERS = 20000
MAX_USERS = 50000
RANDOM_STATE = 42

In [2]:
anime_catalog = pd.read_parquet(ANIME_FILE, columns=["anime_id"])
catalog_ids = set(
    pd.to_numeric(anime_catalog["anime_id"], errors="coerce")
    .dropna()
    .astype("int64")
    .tolist()
)

chunk_reader = pd.read_csv(
    USER_FILE,
    usecols=["user_id", "anime_id", "rating"],
    dtype={"user_id": "int64", "anime_id": "int64", "rating": "int16"},
    chunksize=CHUNK_SIZE,
)

In [3]:
rows_total = 0
rows_after_rating_filter = 0
rows_after_catalog_filter = 0
user_counts = pd.Series(dtype="int64")

for chunk in chunk_reader:
    rows_total += len(chunk)
    chunk = chunk[chunk["rating"] != 0]
    rows_after_rating_filter += len(chunk)

    chunk = chunk[chunk["anime_id"].isin(catalog_ids)]
    rows_after_catalog_filter += len(chunk)

    if chunk.empty:
        continue

    chunk_counts = chunk["user_id"].value_counts()
    user_counts = user_counts.add(chunk_counts, fill_value=0)

In [4]:
user_counts = user_counts.astype("int64").sort_values(ascending=False)
available_users = len(user_counts)

if available_users == 0:
    raise ValueError("No eligible users found after filtering rating != 0 and catalog IDs.")

if available_users <= MIN_USERS:
    target_users = available_users
else:
    target_users = min(MAX_USERS, available_users)

if available_users > target_users:
    n_bins = min(10, int(user_counts.nunique()))
    if n_bins > 1:
        activity_bins = pd.qcut(user_counts, q=n_bins, labels=False, duplicates="drop")
        rng = np.random.default_rng(RANDOM_STATE)
        selected_user_ids = []

        unique_bins = sorted(activity_bins.dropna().unique())
        remaining_slots = target_users

        for i, bin_id in enumerate(unique_bins):
            bin_members = user_counts.index[activity_bins == bin_id]
            bins_left = len(unique_bins) - i
            take = min(len(bin_members), max(1, remaining_slots // bins_left))
            chosen = rng.choice(bin_members.to_numpy(), size=take, replace=False)
            selected_user_ids.extend(chosen.tolist())
            remaining_slots -= take

        if len(selected_user_ids) < target_users:
            selected_index = pd.Index(selected_user_ids)
            additional = user_counts.index.difference(selected_index)[: target_users - len(selected_user_ids)]
            selected_user_ids.extend(additional.tolist())
    else:
        selected_user_ids = user_counts.index[:target_users].tolist()
else:
    selected_user_ids = user_counts.index.tolist()

selected_user_ids = set(selected_user_ids)

In [5]:
clean_chunks = []
chunk_reader = pd.read_csv(
    USER_FILE,
    usecols=["user_id", "anime_id", "rating"],
    dtype={"user_id": "int64", "anime_id": "int64", "rating": "int16"},
    chunksize=CHUNK_SIZE,
)

for chunk in chunk_reader:
    chunk = chunk[chunk["rating"] != 0]
    chunk = chunk[chunk["anime_id"].isin(catalog_ids)]
    chunk = chunk[chunk["user_id"].isin(selected_user_ids)]

    if chunk.empty:
        continue

    chunk = chunk[["user_id", "anime_id", "rating"]].copy()
    chunk["rating"] = chunk["rating"].astype("int8")
    clean_chunks.append(chunk)

if not clean_chunks:
    raise ValueError("No rows remained after applying all filters.")

interactions_clean = pd.concat(clean_chunks, ignore_index=True)
interactions_clean = interactions_clean.astype(
    {"user_id": "int64", "anime_id": "int64", "rating": "int8"}
 )

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
interactions_clean.to_parquet(OUTPUT_FILE, index=False)

rating_drop_pct = (1 - rows_after_rating_filter / rows_total) * 100 if rows_total else 0.0
catalog_drop_pct = (1 - rows_after_catalog_filter / rows_after_rating_filter) * 100 if rows_after_rating_filter else 0.0

print(f"Total rows scanned: {rows_total:,}")
print(f"Rows after rating != 0: {rows_after_rating_filter:,} ({rating_drop_pct:.2f}% dropped)")
print(f"Rows after catalog filter: {rows_after_catalog_filter:,} ({catalog_drop_pct:.2f}% dropped)")
print(f"Available users after filters: {available_users:,}")
print(f"Selected users: {len(selected_user_ids):,}")
print(f"Output rows: {len(interactions_clean):,}")
print(f"Saved: {OUTPUT_FILE}")

Total rows scanned: 109,224,747
Rows after rating != 0: 62,397,712 (42.87% dropped)
Rows after catalog filter: 51,912,145 (16.80% dropped)
Available users after filters: 311,859
Selected users: 50,000
Output rows: 8,370,306
Saved: ../../data/processed/interactions_clean.parquet


## Item Features — 1) Catalog and Interaction Aggregates
Build item-level aggregate statistics from the full interaction dataset using chunked reads.

In [6]:
import ast

ITEM_FEATURES_FILE = DATA_ROOT / "processed"
ARTIFACT_DIR = Path("../../ml/artifacts/content_based_anime_v1")
FEATURE_ARRAYS_FILE = ARTIFACT_DIR / "feature_arrays.npz"
METADATA_FILE = ARTIFACT_DIR / "anime_metadata.csv"

def _safe_numeric(series, dtype="float64"):
    return pd.to_numeric(series, errors="coerce").astype(dtype)

def _parse_genres(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.split(",") if part.strip()]

In [7]:
catalog = pd.read_parquet(
    ANIME_FILE,
    columns=["anime_id", "Score", "Scored By", "Popularity", "Type", "Source", "Genres"],
)

catalog["anime_id"] = _safe_numeric(catalog["anime_id"], dtype="Int64")
catalog = catalog.dropna(subset=["anime_id"]).copy()
catalog["anime_id"] = catalog["anime_id"].astype("int64")
catalog_ids = set(catalog["anime_id"].tolist())

agg = {}
chunk_reader = pd.read_csv(
    USER_FILE,
    usecols=["anime_id", "rating"],
    dtype={"anime_id": "int64", "rating": "int16"},
    chunksize=CHUNK_SIZE,
)

rows_seen = 0
rows_rated = 0
rows_in_catalog = 0

for chunk in chunk_reader:
    rows_seen += len(chunk)
    chunk = chunk[chunk["rating"] != 0]
    rows_rated += len(chunk)

    chunk = chunk[chunk["anime_id"].isin(catalog_ids)]
    rows_in_catalog += len(chunk)

    if chunk.empty:
        continue

    grp = chunk.groupby("anime_id")["rating"].agg(
        rating_count="count",
        rating_sum="sum",
        rating_sq_sum=lambda x: (x.astype("float64") ** 2).sum(),
        positive_count=lambda x: (x >= 7).sum(),
    )

    for anime_id, row in grp.iterrows():
        stats = agg.setdefault(
            int(anime_id),
            {"rating_count": 0, "rating_sum": 0.0, "rating_sq_sum": 0.0, "positive_count": 0},
        )
        stats["rating_count"] += int(row["rating_count"])
        stats["rating_sum"] += float(row["rating_sum"])
        stats["rating_sq_sum"] += float(row["rating_sq_sum"])
        stats["positive_count"] += int(row["positive_count"])

agg_df = pd.DataFrame.from_dict(agg, orient="index")
agg_df.index.name = "anime_id"
agg_df = agg_df.reset_index()

agg_df["avg_user_rating"] = agg_df["rating_sum"] / agg_df["rating_count"]
mean_sq = agg_df["rating_sq_sum"] / agg_df["rating_count"]
variance = (mean_sq - (agg_df["avg_user_rating"] ** 2)).clip(lower=0)
agg_df["rating_std"] = np.sqrt(variance)
agg_df["positive_ratio"] = agg_df["positive_count"] / agg_df["rating_count"]

## Item Features — 2) Join Catalog Features and Encode Categorical Fields
Merge aggregates with catalog metadata, then create type/source encodings and genre vectors.

In [8]:
item_features = catalog.merge(agg_df, on="anime_id", how="left")
item_features["rating_count"] = item_features["rating_count"].fillna(0).astype("int64")
item_features["avg_user_rating"] = item_features["avg_user_rating"].fillna(0.0).astype("float32")
item_features["rating_std"] = item_features["rating_std"].fillna(0.0).astype("float32")
item_features["positive_ratio"] = item_features["positive_ratio"].fillna(0.0).astype("float32")

item_features["catalog_score"] = _safe_numeric(item_features["Score"]).fillna(0.0).astype("float32")
item_features["catalog_scored_by"] = _safe_numeric(item_features["Scored By"]).fillna(0).astype("int64")
item_features["popularity_rank"] = _safe_numeric(item_features["Popularity"]).fillna(0).astype("int64")

item_features["Type"] = item_features["Type"].fillna("unknown").astype(str).str.strip().str.lower()
item_features["Source"] = item_features["Source"].fillna("unknown").astype(str).str.strip().str.lower()
item_features["type_encoded"] = pd.Categorical(item_features["Type"]).codes.astype("int16")
item_features["source_encoded"] = pd.Categorical(item_features["Source"]).codes.astype("int16")

genre_lists = item_features["Genres"].map(_parse_genres)
item_features["n_genres"] = genre_lists.map(len).astype("int16")

unique_genres = sorted({genre for genres in genre_lists for genre in genres})
for genre in unique_genres:
    col = f"genre_{genre.lower().replace(' ', '_').replace('-', '_').replace('/', '_')}"
    item_features[col] = genre_lists.map(lambda values, g=genre: int(g in values)).astype("int8")

## Item Features — 3) Merge Bayesian Scores and Save Artifact
Attach `bayesian_score_norm`, select final columns, and write `item_features.parquet`.

In [9]:
if "bayesian_score_norm" in item_features.columns:
    item_features = item_features.drop(columns=["bayesian_score_norm"])

if METADATA_FILE.exists() and FEATURE_ARRAYS_FILE.exists():
    metadata_columns = set(pd.read_csv(METADATA_FILE, nrows=0).columns)

    if {"anime_id", "anime_index"}.issubset(metadata_columns):
        artifact_meta = pd.read_csv(METADATA_FILE, usecols=["anime_id", "anime_index"])
        artifact_meta["anime_id"] = _safe_numeric(artifact_meta["anime_id"], dtype="Int64")
        artifact_meta["anime_index"] = _safe_numeric(artifact_meta["anime_index"], dtype="Int64")
        artifact_meta = artifact_meta.dropna(subset=["anime_id", "anime_index"]).copy()
        artifact_meta["anime_id"] = artifact_meta["anime_id"].astype("int64")
        artifact_meta["anime_index"] = artifact_meta["anime_index"].astype("int64")
    else:
        artifact_meta = pd.DataFrame(columns=["anime_id", "anime_index"])

    feature_arrays = np.load(FEATURE_ARRAYS_FILE)
    if "bayesian_scores_norm" in feature_arrays.files:
        bayes = feature_arrays["bayesian_scores_norm"]
    elif "bayesian_scores" in feature_arrays.files:
        raw = feature_arrays["bayesian_scores"].astype("float64")
        min_v = np.min(raw)
        max_v = np.max(raw)
        bayes = (raw - min_v) / (max_v - min_v) if max_v > min_v else np.zeros_like(raw)
    else:
        bayes = None

    if bayes is not None and not artifact_meta.empty:
        valid_meta = artifact_meta[artifact_meta["anime_index"].between(0, len(bayes) - 1)].copy()
        bayes_df = valid_meta.copy()
        bayes_df["bayesian_score_norm"] = bayes[bayes_df["anime_index"].to_numpy()].astype("float32")
        item_features = item_features.merge(
            bayes_df[["anime_id", "bayesian_score_norm"]],
            on="anime_id",
            how="left",
        )
    else:
        item_features["bayesian_score_norm"] = np.nan
else:
    item_features["bayesian_score_norm"] = np.nan

if "bayesian_score_norm" not in item_features.columns:
    item_features["bayesian_score_norm"] = np.nan

item_features["bayesian_score_norm"] = item_features["bayesian_score_norm"].fillna(0.0).astype("float32")

keep_cols = [
    "anime_id",
    "avg_user_rating",
    "rating_count",
    "rating_std",
    "positive_ratio",
    "catalog_score",
    "catalog_scored_by",
    "bayesian_score_norm",
    "popularity_rank",
    "type_encoded",
    "source_encoded",
    "n_genres",
] + [c for c in item_features.columns if c.startswith("genre_")]

item_features = item_features[keep_cols].sort_values("anime_id").reset_index(drop=True)

ITEM_FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
item_features.to_csv(ITEM_FEATURES_FILE / "item_features.csv", index=False)
item_features.to_parquet(ITEM_FEATURES_FILE / "item_features.parquet", index=False)
print(f"Rows scanned: {rows_seen:,}")
print(f"Rows with explicit ratings (rating != 0): {rows_rated:,}")
print(f"Rows matching catalog IDs: {rows_in_catalog:,}")
print(f"Item feature shape: {item_features.shape}")
print(f"Saved: {ITEM_FEATURES_FILE}")
print(item_features[["anime_id", "avg_user_rating", "rating_count", "positive_ratio", "bayesian_score_norm"]].head())

Rows scanned: 109,224,747
Rows with explicit ratings (rating != 0): 62,397,712
Rows matching catalog IDs: 51,912,145
Item feature shape: (9185, 28)
Saved: ../../data/processed
   anime_id  avg_user_rating  rating_count  positive_ratio  \
0         1         8.657188         87748        0.939064   
1         5         8.317430         33768        0.937130   
2         6         8.158148         50908        0.904710   
3         7         7.249937         11895        0.739891   
4         8         6.927874          1844        0.651844   

   bayesian_score_norm  
0             0.929268  
1             0.835805  
2             0.820948  
3             0.619680  
4             0.565091  


## User Features (Training Only)
Build user profile aggregates from cleaned interactions and save offline-only features to `data/processed/user_features.parquet`.

In [10]:
import json

USER_FEATURES_FILE = DATA_ROOT / "processed" / "user_features.parquet"

if "interactions_clean" in globals() and isinstance(interactions_clean, pd.DataFrame):
    interactions_for_users = interactions_clean[["user_id", "anime_id", "rating"]].copy()
else:
    interactions_for_users = pd.read_parquet(
        OUTPUT_FILE,
        columns=["user_id", "anime_id", "rating"],
    )

interactions_for_users["user_id"] = pd.to_numeric(interactions_for_users["user_id"], errors="coerce")
interactions_for_users["anime_id"] = pd.to_numeric(interactions_for_users["anime_id"], errors="coerce")
interactions_for_users["rating"] = pd.to_numeric(interactions_for_users["rating"], errors="coerce")
interactions_for_users = interactions_for_users.dropna(subset=["user_id", "anime_id", "rating"]).copy()
interactions_for_users["user_id"] = interactions_for_users["user_id"].astype("int64")
interactions_for_users["anime_id"] = interactions_for_users["anime_id"].astype("int64")
interactions_for_users["rating"] = interactions_for_users["rating"].astype("int8")

user_stats = interactions_for_users.groupby("user_id")["rating"].agg(
    user_avg_rating="mean",
    user_rating_count="count",
    user_rating_std="std",
).reset_index()
user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0.0)

item_meta = pd.read_parquet(
    ANIME_FILE,
    columns=["anime_id", "Genres", "Type", "Popularity"],
)
item_meta["anime_id"] = pd.to_numeric(item_meta["anime_id"], errors="coerce")
item_meta = item_meta.dropna(subset=["anime_id"]).copy()
item_meta["anime_id"] = item_meta["anime_id"].astype("int64")
item_meta["Type"] = item_meta["Type"].fillna("unknown").astype(str).str.strip().str.lower()
item_meta["Popularity"] = pd.to_numeric(item_meta["Popularity"], errors="coerce")

merged = interactions_for_users.merge(
    item_meta,
    on="anime_id",
    how="left",
)

user_popularity = (
    merged.groupby("user_id", as_index=False)["Popularity"]
    .mean()
    .rename(columns={"Popularity": "user_avg_item_popularity"})
)
user_popularity["user_avg_item_popularity"] = user_popularity["user_avg_item_popularity"].fillna(0.0)

merged["genre_list"] = merged["Genres"].map(_parse_genres)
genre_exploded = merged[["user_id", "genre_list"]].explode("genre_list")
genre_exploded = genre_exploded.dropna(subset=["genre_list"])

if genre_exploded.empty:
    user_genre_vector = user_stats[["user_id"]].copy()
    user_genre_vector["user_genre_vector"] = "{}"
else:
    genre_counts = pd.crosstab(genre_exploded["user_id"], genre_exploded["genre_list"])
    genre_dist = genre_counts.div(genre_counts.sum(axis=1), axis=0).fillna(0.0)
    user_genre_vector = genre_dist.apply(
        lambda row: json.dumps({k: float(v) for k, v in row[row > 0].to_dict().items()}),
        axis=1,
    ).reset_index(name="user_genre_vector")

type_frame = merged[["user_id", "Type"]].copy()
type_frame["Type"] = type_frame["Type"].fillna("unknown")
type_counts = pd.crosstab(type_frame["user_id"], type_frame["Type"])
type_dist = type_counts.div(type_counts.sum(axis=1), axis=0).fillna(0.0)
user_type_distribution = type_dist.apply(
    lambda row: json.dumps({k: float(v) for k, v in row[row > 0].to_dict().items()}),
    axis=1,
).reset_index(name="user_type_distribution")

user_features = user_stats.merge(user_popularity, on="user_id", how="left")
user_features = user_features.merge(user_genre_vector, on="user_id", how="left")
user_features = user_features.merge(user_type_distribution, on="user_id", how="left")

user_features["user_avg_item_popularity"] = user_features["user_avg_item_popularity"].fillna(0.0).astype("float32")
user_features["user_genre_vector"] = user_features["user_genre_vector"].fillna("{}")
user_features["user_type_distribution"] = user_features["user_type_distribution"].fillna("{}")
user_features["user_avg_rating"] = user_features["user_avg_rating"].astype("float32")
user_features["user_rating_count"] = user_features["user_rating_count"].astype("int64")
user_features["user_rating_std"] = user_features["user_rating_std"].astype("float32")

user_features = user_features[[
    "user_id",
    "user_avg_rating",
    "user_rating_count",
    "user_rating_std",
    "user_genre_vector",
    "user_type_distribution",
    "user_avg_item_popularity",
]].sort_values("user_id").reset_index(drop=True)

USER_FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
user_features.to_parquet(USER_FEATURES_FILE, index=False)

print(f"User feature shape: {user_features.shape}")
print(f"Saved: {USER_FEATURES_FILE}")
print(user_features.head())

User feature shape: (50000, 7)
Saved: ../../data/processed/user_features.parquet
   user_id  user_avg_rating  user_rating_count  user_rating_std  \
0        1         8.091743                109         1.182756   
1       27         7.604167                 96         1.922056   
2       30         7.440000                 50         0.907115   
3       42         7.708609                302         1.551291   
4       45         7.619718                 71         1.345534   

                                   user_genre_vector  \
0  {"Action": 0.19172932330827067, "Adventure": 0...   
1  {"Action": 0.16538461538461538, "Adventure": 0...   
2  {"Action": 0.27388535031847133, "Adventure": 0...   
3  {"Action": 0.16289592760180996, "Adventure": 0...   
4  {"Action": 0.09803921568627451, "Adventure": 0...   

                              user_type_distribution  user_avg_item_popularity  
0  {"movie": 0.1559633027522936, "ova": 0.0458715...                442.908264  
1  {"movie": 0.25